# Data and calculations

A typical evaluation, step by step: load a CSV, calculate columns, propagate uncertainties,
take a mean, write tables. The comment behind a line says what it does.

In [1]:
import numpy as np
from pathlib import Path

from pemlab import NumberStyle, Project

lab = Project("pemlab_build", tex_prefix="pemlab_build")  # everything ends up in this folder
lab.column_names_from_folder("data")                      # one variable per column to copy (once)

# resistor.csv
U = "U [V]"
I = "I [mA]"


In [2]:
resistor = lab.edit_csv("data/resistor.csv", name="resistor")  # the file itself is never changed
resistor.df.head(3)                                            # the table as pandas shows it

,U [V],I [mA]
0,0.943,2.0
1,1.870,4.0
2,2.828,6.0


## Columns from functions

In [3]:
# The arguments of the function are columns: U is "U [V]", I is "I [A]" (matched by name).
resistor.replace_column("I [mA]", "I [A]", lambda I: I * 1e-3)       # change the unit in place
resistor.add_column("R [Ohm]", lambda U, I: U / I, show=True)        # show=True: listed in the report
resistor.add_column("P [mW]", lambda U, I: 1e3 * U * I,              # numbers= is the look of
                    numbers=NumberStyle(decimals=2))                 # this column in the table
resistor.add_column("P [dBm]", lambda P: 10 * np.log10(P))           # numpy works, P is "P [mW]"
resistor.df.head(3)

,U [V],I [A],R [Ohm],P [mW],P [dBm]
0,0.943,0.002,471.500000,1.886,2.755417
1,1.870,0.004,467.500000,7.480,8.739016
2,2.828,0.006,471.333333,16.968,12.296307


In [4]:
# A column that has another name than the argument is assigned: argument="column name"
resistor.add_column("G [S]", lambda U, current: current / U, current="I [A]")

# A def works the same way (a single `return` at the end is what the report shows)
def conductance_in_mS(R):
    return 1e3 / R

resistor.add_column("G [mS]", conductance_in_mS)

# The formula can still be text
resistor.add_column("P2 [mW]", "1e3 * U * I")
resistor.df.head(3)

,U [V],I [A],R [Ohm],P [mW],P [dBm],G [S],G [mS],P2 [mW]
0,0.943,0.002,471.500000,1.886,2.755417,0.002121,2.120891,1.886
1,1.870,0.004,467.500000,7.480,8.739016,0.002139,2.139037,7.480
2,2.828,0.006,471.333333,16.968,12.296307,0.002122,2.121641,16.968


## Uncertainties

In [5]:
# Uncertainty of the voltage from the accuracy of the multimeter, as a column
resistor.add_column("sigma_U [V]", lambda U: 0.005 * U + 0.002)

# Gaussian propagation, derived analytically: every argument needs a sigma
# (a number for a constant uncertainty, or the name of a column)
resistor.add_uncertainty(
    "sigma_R [Ohm]", lambda U, I: U / I,
    {"U": "sigma_U [V]", "I": 5e-6},
    show=True,
)
resistor.df[["R [Ohm]", "sigma_R [Ohm]"]].head(3)

,R [Ohm],sigma_R [Ohm]
0,471.500000,3.558407
1,467.500000,2.897050
2,471.333333,2.718524


In [6]:
# Values you computed yourself (e.g. fit results) can be added as a column too
resistor.set_column("U_fit [V]", resistor.column("I [A]") * 470.0)
resistor

,U [V],I [A],R [Ohm],P [mW],P [dBm],G [S],G [mS],P2 [mW],sigma_U [V],sigma_R [Ohm],U_fit [V]
0,0.943,0.002,471.500000,1.886,2.755417,0.002121,2.120891,1.886,0.006715,3.558407,0.94
1,1.870,0.004,467.500000,7.480,8.739016,0.002139,2.139037,7.480,0.011350,2.897050,1.88
2,2.828,0.006,471.333333,16.968,12.296307,0.002122,2.121641,16.968,0.016140,2.718524,2.82
3,3.769,0.008,471.125000,30.152,14.793161,0.002123,2.122579,30.152,0.020845,2.622210,3.76
4,4.680,0.010,468.000000,46.800,16.702459,0.002137,2.136752,46.800,0.025400,2.550756,4.70
5,5.627,0.012,468.916667,67.524,18.294582,0.002133,2.132575,67.524,0.030135,2.518839,5.64
6,6.581,0.014,470.071429,92.134,19.644199,0.002127,2.127336,92.134,0.034905,2.498860,6.58
7,7.517,0.016,469.812500,120.272,20.801645,0.002129,2.128509,120.272,0.039585,2.478415,7.52
8,8.460,0.018,470.000000,152.280,21.826429,0.002128,2.127660,152.280,0.044300,2.464571,8.46
9,9.391,0.020,469.550000,187.820,22.737418,0.002130,2.129699,187.820,0.048955,2.450563,9.40


## Mean and tables

In [7]:
analysis = lab.analysis(resistor)
mean = analysis.mean("R_mean", "R [Ohm]", sigma="sigma_R [Ohm]", reference=470)  # weighted mean
mean.mean, mean.error

n = 10
weighted mean = 469.72 ± 0.83
standard deviation = 1.35
chi²/ndf = 0.23
deviation from 470 = -0.282, -0.06 %, 0.34 sigma


(469.71782254376365, 0.8320981321595167)

In [8]:
# A table for every csv file below data/; the .tex files get the same folder structure
lab.tables_from_folder("data")
lab.build()                                               # writes everything, nothing before

def tree(folder):
    for path in sorted(Path(folder).rglob("*")):
        if path.is_file() and path.name != ".pemlab-build":
            print(path.relative_to(folder).as_posix())

tree("pemlab_build")

data/resistor/resistor.calc.json
data/resistor/resistor.csv
data/resistor/resistor.tex
means/R_mean/R_mean.tex
tables/filter/highpass.tex
tables/filter/lowpass.tex
tables/filter/phase.tex
tables/resistor.tex


In [9]:
# What LaTeX includes: \input{pemlab_build/tables/filter/lowpass.tex}
print(Path("pemlab_build/data/resistor/resistor.tex").read_text(encoding="utf-8")[:900])

\begin{table}[H]
  \centering
  \begin{tabular}{@{}ccccccccccc@{}}
    \toprule
    U [V] & I [A] & R [$\Omega$] & P [mW] & P [dBm] & G [S] & G [mS] & P2 [mW] & sigma\_U [V] & sigma\_R [$\Omega$] & U\_fit [V] \\
    \midrule
    0.943 & \num{2e-3} & 471.5 & 1.89 & 2.755 & \num{2.121e-3} & 2.121 & 1.886 & \num{6.715e-3} & 3.558 & 0.9400 \\
    1.87 & \num{4e-3} & 467.5 & 7.48 & 8.739 & \num{2.139e-3} & 2.139 & 7.480 & 0.01135 & 2.897 & 1.880 \\
    2.828 & \num{6e-3} & 471.3 & 16.97 & 12.30 & \num{2.122e-3} & 2.122 & 16.97 & 0.01614 & 2.719 & 2.820 \\
    3.769 & \num{8e-3} & 471.1 & 30.15 & 14.79 & \num{2.123e-3} & 2.123 & 30.15 & 0.02084 & 2.622 & 3.760 \\
    4.68 & 0.01 & 468.0 & 46.80 & 16.70 & \num{2.137e-3} & 2.137 & 46.80 & 0.02540 & 2.551 & 4.700 \\
    5.627 & 0.012 & 468.9 & 67.52 & 18.29 & \num{2.133e-3} & 2.133 & 67.52 & 0.03013 & 2.519 & 5.640 \\
    6.581 & 0.014 & 470.1 & 
